In [0]:
#%run ./transform_data ----- A décommmenter pour lancer les notebooks séparements

dim_calendar_months

In [0]:
# Créer une session Spark
spark = SparkSession.builder.getOrCreate()

# Données brutes
months_data = [
    (1, "Janvier", "January"),
    (2, "Février", "February"),
    (3, "Mars", "March"),
    (4, "Avril", "April"),
    (5, "Mai", "May"),
    (6, "Juin", "June"),
    (7, "Juillet", "July"),
    (8, "Août", "August"),
    (9, "Septembre", "September"),
    (10, "Octobre", "October"),
    (11, "Novembre", "November"),
    (12, "Décembre", "December")
]

# Création du DataFrame
df_months = spark.createDataFrame(months_data, ["month_num", "month_fr", "month_eng"])


df_months = df_months.withColumn("month_num", F.col("month_num").cast("int"))

dim_calendar_months = df_months

Import dim_calendar_months

In [0]:
current_process= "dim_calendar_months"

In [0]:
target_dim_calendar_months = current_catalog +"."+current_schema+"."+current_process
print(target_dim_calendar_months)

In [0]:
all_columns =  dim_calendar_months.columns
display(all_columns)

In [0]:
# define the primary key 
primary_key = [    
    'month_num']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    dim_calendar_months, 
    target_dim_calendar_months, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode  # Use "update" for update mode, "full" for delete/insert mode
    )

dim_calendar

In [0]:

# Paramètre
date_debut = "2025-01-01"

dim_calendar = (
    spark.createDataFrame([(1,)], ["dummy"])
    .select(
        F.explode(
            F.sequence(
                F.to_date(F.lit(date_debut)),
                F.current_date(),
                F.expr("interval 1 day")
            )
        ).alias("day_date")
    )
    .withColumn("month_name", F.date_format("day_date", "MMMM"))
    .withColumn("year_name", F.year("day_date"))
    .orderBy("day_date")
)

dim_calendar = dim_calendar.alias("a").join(
    df_months.alias("b"),
    F.col("a.month_name") == F.col("b.month_eng"),
    "left"
).select(
    "a.*",
    "b.month_num"
)

In [0]:
current_process= "dim_calendar"

In [0]:
target_dim_calendar = current_catalog +"."+current_schema+"."+current_process
print(target_dim_calendar)

In [0]:
all_columns =  dim_calendar.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'day_date']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    dim_calendar, 
    target_dim_calendar, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode  # Use "update" for update mode, "full" for delete/insert mode
    )